# 01 GBDT
## 1 首先specify hyperparameters, 根据
《Comparing hyperparameter tuning methods in machine learning based urban building energy modeling: A study in Chicago - ScienceDirect》


In [17]:
import geopandas as gpd
import pandas as pd
import numpy as np
import os
import re
import matplotlib.pyplot as plt
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import RandomizedSearchCV
from sklearn.inspection import PartialDependenceDisplay
from sklearn.model_selection import train_test_split
from sklearn.model_selection import RepeatedKFold
from scipy.stats import randint, uniform, loguniform

# 📁 文件夹路径
grid_folder = r'D:\seoul\grids\lst_map'

# 🔧 变量定义
target_vars = ['nor_2020', 'ext_2020', 'hr_2020']
explanatory_vars = ['BCR(%)', 'BHV', 'NDVI', 'SVF', 'EV(m)',
                    'Dist_BP', 'Dist_MT', 'Dist_WB', 'WR(%)']

# 📊 保存结果
all_results = []
pdp_records = []
r2_comparison = []

# 🔍 Randomized Search 的分布（based on q0.05–q0.95）
param_dist = {
    # n_estimators = n_rounds(R) in xgboost-
    'n_estimators': [4168],
    # 0000000 02 learning_rate = eta in xgboost
    'learning_rate': loguniform(0.002, 0.355),
    # 0000000 04 subsample = subsample in xgboost# [loc, loc+scale] = [0.545,0.958] loc = 0.545, scale = 0.413
    'subsample': uniform(0.545, 0.413),
    # 0000000  00 max_depth = Def.O = 13  5.6-14 -> 5-14
    ###'max_depth' : uniform (5, 9),

    # min_samples_split ≈ min_child_weight in xgboost, [loc, loc+scale] = [1.295,6.984] loc = 1.295, scale = 5.689
    ''''min_samples_split': uniform (1.295, 5.689),'''
    # max_features ≈ colsample_bytree in xgboost, [loc, loc+scale] = [0.419, 0.864] loc = 0.419, scale = 0.445
    ''''max_features': uniform(0.419, 0.445),'''
    # ccp_alpha = alpha in xgboost
    'ccp_alpha' : [1.113]
}

# === 主循环 ===
for filename in os.listdir(grid_folder):
    if filename.endswith('_clean.shp'):
        input_path = os.path.join(grid_folder, filename)
        match = re.search(r'(\d{3,5})m', filename)
        grid_size = match.group(1)

        gdf = gpd.read_file(input_path)
        gdf_clean = gdf.replace([np.inf, -np.inf], np.nan).dropna(subset=target_vars + explanatory_vars)

        for target in target_vars:
            X = gdf_clean[explanatory_vars]
            y = gdf_clean[target]

            # 🔀 数据划分
            X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

            '''
            # 换成random search  定义log distribution
            # ----> 换成repeated cv ---- #
            # 至少20次 repeated number
            '''
            gbdt = GradientBoostingRegressor(random_state=0)
            cv = RepeatedKFold(n_splits=5, n_repeats=4, random_state=0)
            search = RandomizedSearchCV(
                estimator=gbdt,
                param_distributions=param_dist,
                n_iter= 200,
                scoring='r2',
                cv=cv,
                verbose=2,
                n_jobs=-1,
                random_state=0
            )

            search.fit(X_train, y_train)

            # ✅ 使用测试集评估
            best_model = search.best_estimator_
            y_train_pred = best_model.predict(X_train)
            y_test_pred = best_model.predict(X_test)

            r2_train = best_model.score(X_train, y_train)
            r2_test = r2_score(y_test, y_test_pred)

            rmse_train = np.sqrt(mean_squared_error(y_train, y_train_pred))
            rmse_test = np.sqrt(mean_squared_error(y_test, y_test_pred))

            print(f"✅ {filename} | {target} 最佳参数: {search.best_params_} | R²_train={r2_train:.3f} | R²_test={r2_test:.3f}")

            for var, importance in zip(explanatory_vars, best_model.feature_importances_):
                all_results.append({
                    'GridSize': grid_size,
                    'Target': target,
                    'Feature': var,
                    'FeatureImportance_TrainModel': round(importance, 4),
                    'Train_R2': round(r2_train, 4),
                    'Train_RMSE': round(rmse_train, 4),
                    'Test_R2': round(r2_test, 4),
                    'Test_RMSE': round(rmse_test, 4),
                    **search.best_params_
                })
                r2_comparison.append({
                    'GridSize': grid_size,
                    'Target': target,
                    'Train_R2': round(r2_train, 4),
                    'Test_R2': round(r2_test, 4),
                    'Train_RMSE': round(rmse_train, 4),
                    'Test_RMSE': round(rmse_test, 4)
                })

            # 📈 PDP 提取（Top N 变量）
            sorted_idx = np.argsort(best_model.feature_importances_)[::-1]
            top_features = [explanatory_vars[i] for i in sorted_idx[:9]]

            for feature in top_features:
                # 1) 新建一个仅用来提取 PDP 数据的 fig/ax，然后马上关闭
                fig, ax = plt.subplots()
                disp = PartialDependenceDisplay.from_estimator(best_model, X, [feature], ax=ax) # 使用X数据
                x_vals = disp.lines_[0][0].get_xdata()
                y_vals = disp.lines_[0][0].get_ydata()
                plt.close(fig)
                pdp_records.append({
                    'Feature': feature,
                    'GridSize': grid_size,
                    'Target': target,
                    'X': x_vals,
                    'Y': y_vals
                })

# 保存模型训练后的结果
df_all = pd.DataFrame(all_results)
df_all.to_excel(os.path.join(grid_folder, 'GBDT_Random_Search_Results.xlsx'), index=False)
df_r2 = pd.DataFrame(r2_comparison)
df_r2 = df_r2.sort_values(['Target', 'GridSize'])
df_r2.to_excel(os.path.join(grid_folder, 'R2_Comparison_Train_vs_Test.xlsx'), index=False)
print("✅ R² train vs test comparison saved.")
# 保存 PDP 数据
pdp_df = pd.DataFrame(pdp_records)
pdp_df.to_pickle(os.path.join(grid_folder, 'pdp_records.pkl'))  # 用 pickle 保留 numpy 数组

Fitting 20 folds for each of 50 candidates, totalling 1000 fits


KeyboardInterrupt: 

In [11]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# === 文件路径 ===
grid_folder = r'D:\seoul\grids\lst_map'
input_path = os.path.join(grid_folder, 'R2_Comparison_Train_vs_Test.xlsx')
output_dir = os.path.join(grid_folder, 'figures', 'r2_comparison')
os.makedirs(output_dir, exist_ok=True)

# === 读取数据 ===
df = pd.read_excel(input_path)
df['GridSize'] = df['GridSize'].astype(int)

# === 绘图：每个 Target 一个对比图 ===
for target in df['Target'].unique():
    df_sub = df[df['Target'] == target].sort_values('GridSize')

    plt.figure(figsize=(8, 5))
    plt.plot(df_sub['GridSize'], df_sub['Train_R2'], marker='o', label='Train R²')
    plt.plot(df_sub['GridSize'], df_sub['Test_R2'], marker='o', label='Test R²')

    plt.title(f'R² Comparison — {target}')
    plt.xticks(sorted(df['GridSize'].unique()))  # ✅ 强制显示真实 X 值
    plt.xlabel('Grid Size (m)')
    plt.ylabel('R² Score')
    plt.ylim(0, 1.05)
    plt.grid(True)
    plt.legend()
    plt.tight_layout()

    out_path = os.path.join(output_dir, f'R2_Train_vs_Test_{target}.png')
    plt.savefig(out_path, dpi=300)
    plt.close()
    print(f"📈 Saved R² comparison: {out_path}")


📈 Saved R² comparison: D:\seoul\grids\lst_map\figures\r2_comparison\R2_Train_vs_Test_ext_2020.png
📈 Saved R² comparison: D:\seoul\grids\lst_map\figures\r2_comparison\R2_Train_vs_Test_hr_2020.png
📈 Saved R² comparison: D:\seoul\grids\lst_map\figures\r2_comparison\R2_Train_vs_Test_nor_2020.png


In [4]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import os

# 路径
grid_folder = r'D:\seoul\grids\lst_map'
fig_output_dir = os.path.join(grid_folder, 'figures', 'pdp_gridsearch_combined')
os.makedirs(fig_output_dir, exist_ok=True)

# 读取 PDP 数据
pdp_df = pd.read_pickle(os.path.join(grid_folder, 'pdp_records.pkl'))  # 之前保存的 pickle

# 画图
for (feature, target), group in pdp_df.groupby(['Feature', 'Target']):
    plt.figure(figsize=(7, 8))
    group_sorted = group.copy()
    group_sorted['GridSize'] = group_sorted['GridSize'].astype(int)
    group_sorted = group_sorted.sort_values('GridSize')
    colors = cm.viridis(np.linspace(0, 1, len(group_sorted)))

    for idx, (_, row) in enumerate(group_sorted.iterrows()):
        gridsize = row['GridSize']
        if gridsize == 450:  # 就是450
            plt.plot(row['X'], row['Y'], label=f"{gridsize}m (baseline)", color='red', linewidth=3, linestyle='-')
        else:
            plt.plot(row['X'], row['Y'], label=f"{gridsize}m", color=colors[idx], linewidth=2)

    plt.title(f"{feature} — {target}")
    plt.xlabel(feature)
    plt.ylabel(f"Partial dependence on {target}")
    plt.legend(title='Grid Size')
    plt.tight_layout()
    plt.grid(True)

    fig_path = os.path.join(fig_output_dir, f'pdp_{target}_{feature}_allgrids.png')
    plt.savefig(fig_path, dpi=300)
    plt.close()
    print(f"📊 Saved PDP: {fig_path}")

📊 Saved PDP: D:\seoul\grids\lst_map\figures\pdp_gridsearch_combined\pdp_ext_2020_BCR(%)_allgrids.png
📊 Saved PDP: D:\seoul\grids\lst_map\figures\pdp_gridsearch_combined\pdp_hr_2020_BCR(%)_allgrids.png
📊 Saved PDP: D:\seoul\grids\lst_map\figures\pdp_gridsearch_combined\pdp_nor_2020_BCR(%)_allgrids.png
📊 Saved PDP: D:\seoul\grids\lst_map\figures\pdp_gridsearch_combined\pdp_ext_2020_BHV_allgrids.png
📊 Saved PDP: D:\seoul\grids\lst_map\figures\pdp_gridsearch_combined\pdp_hr_2020_BHV_allgrids.png
📊 Saved PDP: D:\seoul\grids\lst_map\figures\pdp_gridsearch_combined\pdp_nor_2020_BHV_allgrids.png
📊 Saved PDP: D:\seoul\grids\lst_map\figures\pdp_gridsearch_combined\pdp_ext_2020_Dist_BP_allgrids.png
📊 Saved PDP: D:\seoul\grids\lst_map\figures\pdp_gridsearch_combined\pdp_hr_2020_Dist_BP_allgrids.png
📊 Saved PDP: D:\seoul\grids\lst_map\figures\pdp_gridsearch_combined\pdp_nor_2020_Dist_BP_allgrids.png
📊 Saved PDP: D:\seoul\grids\lst_map\figures\pdp_gridsearch_combined\pdp_ext_2020_Dist_MT_allgrids.pn

In [7]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# 📁 路径设置
grid_folder = r'D:\seoul\grids\lst_map'
input_path = os.path.join(grid_folder, 'GBDT_GridSearch_Results.xlsx')
output_dir = os.path.join(grid_folder, 'figures', 'grid_comparison')
os.makedirs(output_dir, exist_ok=True)

# 📄 读取数据
df = pd.read_excel(input_path)
df['GridSize'] = df['GridSize'].astype(int)

# ✅ 提取每个 GridSize + Target 的最佳参数（去重）
best_params = df.drop_duplicates(subset=['GridSize', 'Target'])[
    ['GridSize', 'Target', 'learning_rate', 'max_depth', 'n_estimators', 'subsample']
].sort_values(['Target', 'GridSize'])

# 💾 保存为单独 Excel
param_path = os.path.join(grid_folder, 'GBDT_BestParams_byGrid.xlsx')
best_params.to_excel(param_path, index=False)
print(f"✅ 已保存最优参数表：{param_path}")

# 📈 R² 和 RMSE 折线图
for metric in ['R2', 'RMSE']:
    plt.figure(figsize=(8, 5))
    for target in df['Target'].unique():
        sub_df = df[df['Target'] == target]
        sub_df = sub_df.groupby('GridSize')[metric].mean().reset_index()

        # 👉 取这个目标变量的参数示例（第一个 GridSize）
        example_param = best_params[best_params['Target'] == target].iloc[0]
        param_text = f"lr={example_param['learning_rate']}, depth={example_param['max_depth']}, est={example_param['n_estimators']}"

        plt.plot(sub_df['GridSize'], sub_df[metric], marker='o', label=f"{target} ({param_text})")

    plt.title(f'{metric} vs Grid Size (Best GBDT via GridSearchCV)')
    plt.xlabel('Grid Size (m)')
    plt.ylabel(metric)
    plt.grid(True)
    plt.legend(title='Target + Params', fontsize=9)
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, f'{metric}_Line_by_GridSize_withParams.png'), dpi=300)
    plt.close()

print(f"✅ 折线图已保存至：{output_dir}")


# 🔥 2. 特征重要性热力图（每个变量 × grid size，按平均值）
pivot = df.pivot_table(index='Feature', columns='GridSize', values='Importance', aggfunc='mean')
plt.figure(figsize=(12, 6))
sns.heatmap(pivot, cmap='YlOrBr', annot=True, fmt=".2f")
plt.title('Feature Importance (Mean) by Grid Size')
plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'FeatureImportance_Heatmap_by_Grid.png'), dpi=300)
plt.close()

# 📈 3. 每个 Feature 的重要性随 GridSize 变化线图（按目标变量分别画）
for target in df['Target'].unique():
    df_target = df[df['Target'] == target]
    grouped = df_target[['GridSize', 'Feature', 'Importance']].copy()

    plt.figure(figsize=(5, 6))
    sns.lineplot(data=grouped, x='GridSize', y='Importance', hue='Feature', marker='o')

    plt.title(f'Feature Importance per Grid Size — {target}')
    plt.xlabel('Grid Size (m)')
    plt.ylabel('Feature Importance')
    plt.xticks(sorted(df_target['GridSize'].unique()))  # ✅ 强制显示真实 X 值
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, f'ImportanceTrend_{target}.png'), dpi=300)
    plt.close()


print(f"✅ 所有按 Grid 对比图保存至：{output_dir}")


✅ 已保存最优参数表：D:\seoul\grids\lst_map\GBDT_BestParams_byGrid.xlsx
✅ 折线图已保存至：D:\seoul\grids\lst_map\figures\grid_comparison
✅ 所有按 Grid 对比图保存至：D:\seoul\grids\lst_map\figures\grid_comparison


In [6]:
import matplotlib.pyplot as plt
import seaborn as sns

param_cols = ['learning_rate', 'max_depth', 'n_estimators', 'subsample']
df_params = df.drop_duplicates(subset=['GridSize', 'Target'] + param_cols).copy()
df_params['GridSize'] = df_params['GridSize'].astype(int)
df_params = df_params.sort_values(['Target', 'GridSize'])

# 创建输出路径
output_dir = os.path.join(grid_folder, 'figures', 'hyperparam_analysis')
os.makedirs(output_dir, exist_ok=True)

# === 折线图：连续型参数趋势 ===
for param in ['learning_rate', 'subsample']:
    plt.figure(figsize=(8, 5))
    for target in df_params['Target'].unique():
        df_sub = df_params[df_params['Target'] == target]
        plt.plot(df_sub['GridSize'], df_sub[param], marker='o', label=target)

    plt.title(f'{param} across Grid Sizes')
    plt.xlabel('Grid Size (m)')
    plt.ylabel(param)
    plt.grid(True)
    plt.legend(title='Target')
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, f'{param}_trend.png'), dpi=300)
    plt.close()

# === 柱状图：分类型参数频率 ===
for param in ['max_depth', 'n_estimators']:
    plt.figure(figsize=(10, 5))
    sns.countplot(data=df_params, x='GridSize', hue=param, palette='Set2')
    plt.title(f'{param} distribution across Grid Sizes')
    plt.xlabel('Grid Size (m)')
    plt.ylabel('Count')
    plt.legend(title=param)
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, f'{param}_count.png'), dpi=300)
    plt.close()

print(f"✅ 所有超参数比较图已保存到：{output_dir}")


✅ 所有超参数比较图已保存到：D:\seoul\grids\lst_map\figures\hyperparam_analysis


In [20]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# 读取用户上传的 Excel 文件
input_path = "D:\seoul\grids\lst_map/GBDT_GridSearch_Results.xlsx"
df = pd.read_excel(input_path)

# 转换 GridSize 为整数型，便于排序和绘图
df['GridSize'] = df['GridSize'].astype(int)

# 创建输出文件夹
output_dir = "/mnt/data/gridsearch_param_visualization"
os.makedirs(output_dir, exist_ok=True)

# 筛选唯一的最优参数组合（按 GridSize 和 Target）
best_params = df.drop_duplicates(subset=['GridSize', 'Target'])[
    ['GridSize', 'Target', 'learning_rate', 'max_depth', 'n_estimators', 'subsample']
].sort_values(['Target', 'GridSize'])

# 绘制每个超参数随 GridSize 变化的线图，按目标变量分图
param_list = ['learning_rate', 'max_depth', 'n_estimators', 'subsample']
for param in param_list:
    for target in best_params['Target'].unique():
        subset = best_params[best_params['Target'] == target]
        plt.figure(figsize=(8, 5))
        sns.lineplot(data=subset, x='GridSize', y=param, marker='o')
        plt.title(f"{param} vs GridSize — {target}")
        plt.xlabel("Grid Size (m)")
        plt.ylabel(param)
        plt.grid(True)
        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, f"{param}_vs_GridSize_{target}.png"), dpi=300)
        plt.close()

import ace_tools as tools; tools.display_dataframe_to_user(name="Best GBDT Parameters", dataframe=best_params)


<>:7: SyntaxWarning: invalid escape sequence '\s'
<>:7: SyntaxWarning: invalid escape sequence '\s'
C:\Users\owner\AppData\Local\Temp\ipykernel_27964\3670598890.py:7: SyntaxWarning: invalid escape sequence '\s'
  input_path = "D:\seoul\grids\lst_map/GBDT_GridSearch_Results.xlsx"
C:\Users\owner\AppData\Local\Temp\ipykernel_27964\3670598890.py:7: SyntaxWarning: invalid escape sequence '\s'
  input_path = "D:\seoul\grids\lst_map/GBDT_GridSearch_Results.xlsx"


ModuleNotFoundError: No module named 'ace_tools'